# 2D Diffusion Simulation - Basic Test

**Goal:** Validate core 2D diffusion implementation using MSD analysis

**Tests:**
1. Single molecule diffusion with known D
2. MSD analysis recovers correct D value
3. Multiple molecules with different D values
4. Boundary conditions work correctly

**Expected:** MSD should give D within ±10% of input value

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from DiffusionSimulation import (
    DiffusionSimulator2D,
    compute_msd_from_trajectory,
    estimate_D_from_msd
)
from PlottingBase import PublicationPlotter

plotter = PublicationPlotter()

# Set random seed for reproducibility
np.random.seed(42)

## Test 1: Single Molecule Diffusion

Simulate a single molecule and verify MSD analysis recovers the correct D.

In [ ]:
# Simulation parameters
D_true = 1000.0  # nm²/ms (= 1 μm²/s)
dt = 10.0        # ms timestep
t_exposure = 10.0  # ms exposure
n_steps = 500    # 5 seconds total

# Camera parameters
sigma0 = 20.0    # nm localization error
s0 = 200.0       # nm PSF width

# Simulation area: 10 x 10 μm
area = (10000.0, 10000.0)  # nm

# Create simulator
sim = DiffusionSimulator2D(
    area=area,
    dt=dt,
    t_exposure=t_exposure,
    sigma0=sigma0,
    s0=s0,
    boundary='reflective'
)

# Add single molecule at center
center = np.array(area) / 2
mol = sim.add_molecule('G', center, D_free=D_true)

print(f"Starting simulation...")
print(f"True D = {D_true} nm²/ms = {D_true/1000} μm²/s")
print(f"Timestep = {dt} ms")
print(f"Total time = {n_steps * dt / 1000} s")

# Run simulation
sim.run(n_steps)

print(f"\nSimulation complete!")

In [ ]:
# Get trajectory
trajectory, times = sim.get_trajectory(0)

print(f"Trajectory shape: {trajectory.shape}")
print(f"Number of positions: {len(trajectory)}")

# Plot trajectory
fig, ax = plotter.create_figure(figsize=(6, 6))

# Plot path
ax.plot(trajectory[:, 0]/1000, trajectory[:, 1]/1000, 
        'b-', alpha=0.3, lw=0.5, label='Path')

# Mark start and end
ax.plot(trajectory[0, 0]/1000, trajectory[0, 1]/1000, 
        'go', ms=8, label='Start')
ax.plot(trajectory[-1, 0]/1000, trajectory[-1, 1]/1000, 
        'ro', ms=8, label='End')

ax.set_xlabel('X position (μm)')
ax.set_ylabel('Y position (μm)')
ax.set_title(f'Single Molecule Trajectory (D = {D_true/1000} μm²/s)')
ax.legend()
ax.set_aspect('equal')

plotter.save_or_show(fig)

# Calculate total displacement
total_displacement = np.linalg.norm(trajectory[-1] - trajectory[0])
print(f"\nTotal displacement: {total_displacement:.0f} nm = {total_displacement/1000:.2f} μm")
print(f"Expected RMS displacement: ~{np.sqrt(4 * D_true * n_steps * dt):.0f} nm")

## Test 2: MSD Analysis

Compute MSD and fit to recover diffusion coefficient.

In [ ]:
# Compute MSD
tau_array, msd_array = compute_msd_from_trajectory(trajectory, max_tau=100)

# Convert tau to real time
time_lags = tau_array * dt  # ms

# Estimate D from MSD
D_estimated = estimate_D_from_msd(tau_array, msd_array, dt, n_d=2, fit_points=20)

print(f"True D:      {D_true:.2f} nm²/ms = {D_true/1000:.3f} μm²/s")
print(f"Estimated D: {D_estimated:.2f} nm²/ms = {D_estimated/1000:.3f} μm²/s")
print(f"Error:       {abs(D_estimated - D_true)/D_true * 100:.1f}%")
print(f"\nWithin ±10%: {'✓ PASS' if abs(D_estimated - D_true)/D_true < 0.10 else '✗ FAIL'}")

In [ ]:
# Plot MSD vs time
fig, ax = plotter.create_figure()

# Plot measured MSD
ax.plot(time_lags, msd_array, 'o', ms=4, alpha=0.6, label='Measured MSD')

# Plot theoretical MSD for true D
msd_true = 4 * D_true * time_lags
ax.plot(time_lags, msd_true, 'g--', lw=2, label=f'Theory (D={D_true/1000:.2f} μm²/s)')

# Plot fitted MSD
msd_fit = 4 * D_estimated * time_lags
ax.plot(time_lags, msd_fit, 'r-', lw=2, label=f'Fit (D={D_estimated/1000:.2f} μm²/s)')

ax.set_xlabel('Time lag (ms)')
ax.set_ylabel('MSD (nm²)')
ax.set_title('Mean Squared Displacement Analysis')
ax.legend()

plotter.save_or_show(fig)

## Test 3: Multiple Molecules with Same D

Simulate multiple molecules and average their MSDs.

In [ ]:
# Create new simulator
sim_multi = DiffusionSimulator2D(
    area=area,
    dt=dt,
    t_exposure=t_exposure,
    sigma0=sigma0,
    s0=s0,
    boundary='reflective'
)

# Add 50 molecules with random positions
n_molecules = 50
molecules = sim_multi.add_molecules_random(
    n_molecules=n_molecules,
    color='G',
    D_free=D_true
)

print(f"Added {n_molecules} molecules with D = {D_true/1000} μm²/s")

# Run simulation
print(f"Running simulation...")
sim_multi.run(n_steps)
print(f"Complete!")

In [ ]:
# Get all trajectories
all_trajectories = sim_multi.get_all_trajectories()

# Plot all trajectories
fig, ax = plotter.create_figure(figsize=(6, 6))

for mol_id, (traj, times) in all_trajectories.items():
    ax.plot(traj[:, 0]/1000, traj[:, 1]/1000, 
            '-', alpha=0.2, lw=0.5)

ax.set_xlabel('X position (μm)')
ax.set_ylabel('Y position (μm)')
ax.set_title(f'{n_molecules} Molecules Diffusing (D = {D_true/1000} μm²/s)')
ax.set_aspect('equal')
ax.set_xlim(0, area[0]/1000)
ax.set_ylim(0, area[1]/1000)

plotter.save_or_show(fig)

In [ ]:
# Compute MSD for each molecule and average
all_msds = []
all_D_estimates = []

for mol_id, (traj, times) in all_trajectories.items():
    tau, msd = compute_msd_from_trajectory(traj, max_tau=100)
    all_msds.append(msd)
    
    D_est = estimate_D_from_msd(tau, msd, dt, n_d=2, fit_points=20)
    all_D_estimates.append(D_est)

# Average MSD
avg_msd = np.mean(all_msds, axis=0)
std_msd = np.std(all_msds, axis=0)

# Average D estimate
D_avg = np.mean(all_D_estimates)
D_std = np.std(all_D_estimates)

print(f"Results from {n_molecules} molecules:")
print(f"="*50)
print(f"True D:      {D_true:.2f} nm²/ms = {D_true/1000:.3f} μm²/s")
print(f"Average D:   {D_avg:.2f} ± {D_std:.2f} nm²/ms")
print(f"           = {D_avg/1000:.3f} ± {D_std/1000:.3f} μm²/s")
print(f"Error:       {abs(D_avg - D_true)/D_true * 100:.1f}%")
print(f"\nWithin ±10%: {'✓ PASS' if abs(D_avg - D_true)/D_true < 0.10 else '✗ FAIL'}")

In [ ]:
# Plot averaged MSD
fig, ax = plotter.create_figure()

time_lags = tau_array * dt

# Plot average MSD with error bars
ax.errorbar(time_lags, avg_msd, yerr=std_msd, 
            fmt='o', ms=4, alpha=0.6, capsize=3,
            label=f'Average MSD (n={n_molecules})')

# Plot theoretical MSD
msd_true = 4 * D_true * time_lags
ax.plot(time_lags, msd_true, 'g--', lw=2, 
        label=f'Theory (D={D_true/1000:.2f} μm²/s)')

# Plot fitted MSD
msd_fit = 4 * D_avg * time_lags
ax.plot(time_lags, msd_fit, 'r-', lw=2, 
        label=f'Fit (D={D_avg/1000:.2f}±{D_std/1000:.2f} μm²/s)')

ax.set_xlabel('Time lag (ms)')
ax.set_ylabel('MSD (nm²)')
ax.set_title('Average MSD from Multiple Molecules')
ax.legend()

plotter.save_or_show(fig)

In [ ]:
# Plot histogram of D estimates
fig, ax = plotter.create_figure()

ax.hist(np.array(all_D_estimates)/1000, bins=20, alpha=0.7, 
        edgecolor='black', label='Estimated D values')

# Mark true value
ax.axvline(D_true/1000, color='green', linestyle='--', lw=2, 
           label=f'True D = {D_true/1000:.2f} μm²/s')

# Mark average
ax.axvline(D_avg/1000, color='red', linestyle='-', lw=2,
           label=f'Mean = {D_avg/1000:.2f} μm²/s')

ax.set_xlabel('Diffusion Coefficient (μm²/s)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of D Estimates (n={n_molecules})')
ax.legend()

plotter.save_or_show(fig)

## Test 4: Different Diffusion Coefficients

Simulate molecules with different D values and verify each is recovered correctly.

In [ ]:
# Test different D values
D_values = [500.0, 1000.0, 2000.0, 5000.0]  # nm²/ms
colors = ['R', 'G', 'B', 'R']
n_mols_per_type = 20

# Create simulator
sim_varied = DiffusionSimulator2D(
    area=area,
    dt=dt,
    t_exposure=t_exposure,
    sigma0=sigma0,
    s0=s0,
    boundary='reflective'
)

# Add molecules with different D values
mol_types = {}
for i, (D, color) in enumerate(zip(D_values, colors)):
    mols = sim_varied.add_molecules_random(
        n_molecules=n_mols_per_type,
        color=color,
        D_free=D
    )
    mol_types[i] = {'D_true': D, 'color': color, 'mol_ids': [m.molecule_id for m in mols]}
    print(f"Added {n_mols_per_type} {color} molecules with D = {D/1000:.2f} μm²/s")

# Run simulation
print(f"\nRunning simulation...")
sim_varied.run(n_steps)
print(f"Complete!")

In [ ]:
# Analyze each molecule type
results = []

for type_id, info in mol_types.items():
    D_true = info['D_true']
    mol_ids = info['mol_ids']
    color = info['color']
    
    # Get all trajectories for this type
    D_estimates = []
    for mol_id in mol_ids:
        traj, times = sim_varied.get_trajectory(mol_id)
        tau, msd = compute_msd_from_trajectory(traj, max_tau=100)
        D_est = estimate_D_from_msd(tau, msd, dt, n_d=2, fit_points=20)
        D_estimates.append(D_est)
    
    D_avg = np.mean(D_estimates)
    D_std = np.std(D_estimates)
    error_pct = abs(D_avg - D_true) / D_true * 100
    
    results.append({
        'color': color,
        'D_true': D_true,
        'D_avg': D_avg,
        'D_std': D_std,
        'error_pct': error_pct
    })
    
    print(f"{color} molecules (D_true = {D_true/1000:.2f} μm²/s):")
    print(f"  D_estimated = {D_avg/1000:.2f} ± {D_std/1000:.2f} μm²/s")
    print(f"  Error = {error_pct:.1f}%")
    print(f"  Status: {'✓ PASS' if error_pct < 10 else '✗ FAIL'}")
    print()

In [ ]:
# Plot comparison
fig, ax = plotter.create_figure()

D_true_values = [r['D_true']/1000 for r in results]
D_estimated_values = [r['D_avg']/1000 for r in results]
D_errors = [r['D_std']/1000 for r in results]

# Plot estimated vs true
ax.errorbar(D_true_values, D_estimated_values, yerr=D_errors,
            fmt='o', ms=8, capsize=5, label='Estimated')

# Plot perfect agreement line
d_range = [0, max(D_true_values) * 1.1]
ax.plot(d_range, d_range, 'k--', lw=2, alpha=0.5, label='Perfect agreement')

# Plot ±10% error bands
ax.fill_between(d_range, 
                [d * 0.9 for d in d_range], 
                [d * 1.1 for d in d_range],
                alpha=0.2, color='gray', label='±10% error')

ax.set_xlabel('True D (μm²/s)')
ax.set_ylabel('Estimated D (μm²/s)')
ax.set_title('MSD Analysis Validation')
ax.legend()

plotter.save_or_show(fig)

# Print summary
print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)
all_pass = all(r['error_pct'] < 10 for r in results)
print(f"All D values recovered within ±10%: {'✓ PASS' if all_pass else '✗ FAIL'}")
avg_error = np.mean([r['error_pct'] for r in results])
print(f"Average error: {avg_error:.1f}%")

## Summary

This notebook validates the core 2D diffusion simulation by:

1. ✓ Simulating single molecule diffusion with realistic camera effects
2. ✓ Computing MSD from trajectories
3. ✓ Recovering diffusion coefficients from MSD analysis
4. ✓ Testing multiple molecules and averaging
5. ✓ Validating across different D values

**Next steps:**
- Add binding kinetics (Gillespie algorithm)
- Implement multicolor binding rules
- Add camera imaging module